# 3. Скоринг

**4 фактора:** skill, domain, exp, rating (goal убран — вклад <0.5%)

**Нужны:** все файлы из тетрадок 1–2 + `mentor_ratings.csv`

**Создаёт:** `normalization_bounds.csv`, `mentee_weights.csv`, `recommendations.csv`

In [1]:
import csv, math, os, random
from collections import defaultdict, Counter

random.seed(42)
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def parse_vector(s):
    if not s or not s.strip(): return None
    return [float(x) for x in s.split(",")]

def parse_set(s):
    if not s or not s.strip(): return None
    return set(s.split("|"))

def parse_bool(s):
    return str(s).strip().lower() in ("true","1","yes")

DOMAINS = [r["domain_name"].strip()
           for r in load_csv(os.path.join(BASE_DIR,"ontology_domains.csv"))]

SIM_MATRIX = {}
with open(os.path.join(BASE_DIR,"ontology_domain_similarity.csv"),
          newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        d1 = row["domain"].strip()
        for d2 in DOMAINS:
            SIM_MATRIX[(d1,d2)] = float(row.get(d2,0))

ratings_by_id = {r["mentor_id"]: float(r["rating"])
                 for r in load_csv(os.path.join(BASE_DIR,"mentor_ratings.csv"))}

print(f"Областей: {len(DOMAINS)}  |  Рейтингов: {len(ratings_by_id)}")


Областей: 10  |  Рейтингов: 2000


In [2]:
def parse_mentee(r):
    return {
        "id":            r["id"],
        "name":          r["name"],
        "level_score":   int(r["level_score"]),
        "skills_missing": parse_bool(r["skills_missing"]),
        "language_set":  parse_set(r["language_set"]),
        "format_set":    parse_set(r["format_set"]),
        "domain_vector": parse_vector(r["domain_vector"]),
        "skills_list":   [s.strip() for s in
                          r.get("skills_normalized","").split(";") if s.strip()],
    }

def parse_mentor(r):
    return {
        "id":               r["id"],
        "name":             r["name"],
        "profession":       r["profession"],
        "domain":           r.get("domain",""),
        "level_score":      int(r["level_score"]),
        "level_raw":        r.get("level_raw",""),
        "language_set":     parse_set(r["language_set"]),
        "format_set":       parse_set(r["format_set"]),
        "domain_vector":    parse_vector(r["domain_vector"]),
        "skills_list":      [s.strip() for s in
                             r.get("skills_normalized","").split(";") if s.strip()],
        "experience_norm":  float(r["experience_norm"]),
        "experience_years": int(r.get("experience_years",0) or 0),
        "available":        parse_bool(r["available"]),
        "boosted":          parse_bool(r["boosted"]),
        "boost_k":          float(r["boost_k"]),
        "rating":           ratings_by_id.get(r["id"], 3.8),
    }

mentees = [parse_mentee(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentees_processed.csv"))]
mentors = [parse_mentor(r)
           for r in load_csv(os.path.join(BASE_DIR,"mentors_processed.csv"))]

print(f"Mentees: {len(mentees)}  |  Mentors: {len(mentors)}")
print(f"Доступных менторов: {sum(1 for m in mentors if m['available'])}")


Mentees: 5000  |  Mentors: 2000
Доступных менторов: 1613


In [3]:
# ── 4 фактора: skill, domain, exp, rating (goal убран — вклад <0.5%) ─────────

def jaccard(s1, s2):
    a, b = set(s1), set(s2)
    if not a or not b: return None
    return len(a & b) / len(a | b)

def domain_sim_raw(mentee, mentor):
    v1, v2 = mentee["domain_vector"], mentor["domain_vector"]
    if not v1 or not v2: return 0.0
    return sum(
        v1[i] * SIM_MATRIX.get((d1,d2), 0) * v2[j]
        for i,d1 in enumerate(DOMAINS)
        for j,d2 in enumerate(DOMAINS)
    )

def sets_compat(s1, s2):
    if s1 is None or s2 is None: return True
    return len(s1 & s2) > 0

def hard_filter(mentee, pool):
    return [m for m in pool
            if m["available"]
            and m["level_score"] > mentee["level_score"]
            and sets_compat(mentee["language_set"], m["language_set"])
            and sets_compat(mentee["format_set"],   m["format_set"])]

def minmax(v, vmin, vmax):
    if vmax == vmin: return 0.5
    return round(max(0.0, min(1.0, (v - vmin) / (vmax - vmin))), 4)

def compute_score(mentee, mentor, weights):
    # skill: Jaccard. Пустой профиль → 0 (наказание, не исключение)
    sk_raw = jaccard(mentee["skills_list"], mentor["skills_list"])
    do_raw = domain_sim_raw(mentee, mentor)

    skill_val  = minmax(sk_raw, SKILL_MIN, SKILL_MAX) if sk_raw is not None else 0.0
    domain_val = minmax(do_raw, DOMAIN_MIN, DOMAIN_MAX)
    exp_val    = minmax(mentor["experience_norm"], EXP_MIN, EXP_MAX)
    rating_val = minmax((mentor["rating"] - 1) / 4, RATING_MIN, RATING_MAX)

    w_sk = float(weights["w_skills"])
    w_do = float(weights["w_domain"])
    w_ex = float(weights["w_exp"])
    w_ra = float(weights["w_rating"])

    score = w_sk*skill_val + w_do*domain_val + w_ex*exp_val + w_ra*rating_val

    breakdown = {
        "skill":  {"sim":skill_val,  "weight":w_sk,
                   "contribution":round(w_sk*skill_val,4),
                   "penalized": sk_raw is None},
        "domain": {"sim":domain_val, "weight":w_do,
                   "contribution":round(w_do*domain_val,4), "penalized":False},
        "exp":    {"sim":exp_val,    "weight":w_ex,
                   "contribution":round(w_ex*exp_val,4),    "penalized":False},
        "rating": {"sim":rating_val, "weight":w_ra,
                   "contribution":round(w_ra*rating_val,4), "penalized":False},
    }
    return round(score, 4), breakdown

BOOST_THRESHOLD = 0.30
TOP_K = 5

def apply_boost(score, mentor):
    if mentor["boosted"] and score >= BOOST_THRESHOLD:
        return round(score * (1 + mentor["boost_k"]), 4), True
    return score, False

print("Функции скоринга определены (4 фактора: skill, domain, exp, rating)")


Функции скоринга определены (4 фактора: skill, domain, exp, rating)


In [4]:
def pct(vals, p):
    vals = [v for v in vals if v is not None]
    if not vals: return 0.0
    return sorted(vals)[int(len(vals) * p / 100)]

print("Считаем границы нормализации p5-p95 из 300 случайных пар...")
sample = random.sample(mentees, min(300, len(mentees)))
s_vals, d_vals, e_vals, r_vals = [], [], [], []

for mentee in sample:
    cands = [m for m in mentors
             if m["available"] and m["level_score"] > mentee["level_score"]][:50]
    for mentor in cands:
        sk = jaccard(mentee["skills_list"], mentor["skills_list"])
        if sk is not None: s_vals.append(sk)
        d_vals.append(domain_sim_raw(mentee, mentor))
        e_vals.append(mentor["experience_norm"])
        r_vals.append((mentor["rating"] - 1) / 4)

SKILL_MIN,  SKILL_MAX  = pct(s_vals,5), pct(s_vals,95)
DOMAIN_MIN, DOMAIN_MAX = pct(d_vals,5), pct(d_vals,95)
EXP_MIN,    EXP_MAX    = pct(e_vals,5), pct(e_vals,95)
RATING_MIN, RATING_MAX = pct(r_vals,5), pct(r_vals,95)

print(f"  skill:  [{SKILL_MIN:.4f}, {SKILL_MAX:.4f}]")
print(f"  domain: [{DOMAIN_MIN:.4f}, {DOMAIN_MAX:.4f}]")
print(f"  exp:    [{EXP_MIN:.4f}, {EXP_MAX:.4f}]")
print(f"  rating: [{RATING_MIN:.4f}, {RATING_MAX:.4f}]")

with open(os.path.join(BASE_DIR,"normalization_bounds.csv"),
          "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["factor","min","max"])
    w.writeheader()
    for factor,mn,mx in [("skill",SKILL_MIN,SKILL_MAX),
                          ("domain",DOMAIN_MIN,DOMAIN_MAX),
                          ("exp",EXP_MIN,EXP_MAX),
                          ("rating",RATING_MIN,RATING_MAX)]:
        w.writerow({"factor":factor,"min":mn,"max":mx})
print("normalization_bounds.csv создан")


Считаем границы нормализации p5-p95 из 300 случайных пар...
  skill:  [0.0000, 0.3333]
  domain: [0.0335, 0.6874]
  exp:    [0.3608, 1.0000]
  rating: [0.4250, 0.9500]
normalization_bounds.csv создан


In [5]:
# 4 фактора: floor 10% каждый, flex 60% распределяется по приоритетам
FLOOR = 0.10
FLEX  = 1.0 - 4 * FLOOR  # 0.60

def compute_weights(p_sk, p_do, p_ex, p_ra):
    priorities = [p_sk, p_do, p_ex, p_ra]
    total = sum(priorities) or 1
    return [round(FLOOR + FLEX*(p/total),4) for p in priorities]

mentee_weights = []
for m in mentees:
    p_sk = random.randint(1,5)
    p_do, p_ex, p_ra = (random.randint(1,5) for _ in range(3))
    w = compute_weights(p_sk, p_do, p_ex, p_ra)
    mentee_weights.append({
        "mentee_id":m["id"],
        "priority_skills":p_sk,"priority_domain":p_do,
        "priority_exp":p_ex,"priority_rating":p_ra,
        "w_skills":w[0],"w_domain":w[1],"w_exp":w[2],"w_rating":w[3],
    })

weights_by_id = {w["mentee_id"]:w for w in mentee_weights}

with open(os.path.join(BASE_DIR,"mentee_weights.csv"),"w",newline="",encoding="utf-8") as f:
    wcsv = csv.DictWriter(f, fieldnames=["mentee_id","priority_skills","priority_domain",
        "priority_exp","priority_rating","w_skills","w_domain","w_exp","w_rating"])
    wcsv.writeheader()
    wcsv.writerows(mentee_weights)
print(f"mentee_weights.csv создан  ({len(mentee_weights)} строк)")


mentee_weights.csv создан  (5000 строк)


In [6]:
print(f"Генерируем рекомендации для {len(mentees)} менти...")
all_recs = []

for i, mentee in enumerate(mentees):
    if i % 500 == 0: print(f"  {i}/{len(mentees)}...")
    w = weights_by_id[mentee["id"]]
    candidates = hard_filter(mentee, mentors)[:300]  # ограничение для скорости
    results = []
    for mentor in candidates:
        organic, breakdown = compute_score(mentee, mentor, w)
        final, is_boosted  = apply_boost(organic, mentor)
        top_factor = max(breakdown, key=lambda k: breakdown[k]["contribution"])
        penalized  = any(v["penalized"] for v in breakdown.values())
        results.append({
            "mentee_id":mentee["id"],"rank":0,
            "mentor_id":mentor["id"],"mentor_name":mentor["name"],
            "profession":mentor["profession"],"mentor_rating":mentor["rating"],
            "organic_score":organic,"final_score":final,
            "is_boosted":is_boosted,"top_factor":top_factor,
            "profile_penalized":penalized,
            "w_skill_contrib":  breakdown["skill"]["contribution"],
            "w_domain_contrib": breakdown["domain"]["contribution"],
            "w_exp_contrib":    breakdown["exp"]["contribution"],
            "w_rating_contrib": breakdown["rating"]["contribution"],
        })
    results.sort(key=lambda x: -x["final_score"])
    for rank, r in enumerate(results[:TOP_K],1):
        r["rank"] = rank
        all_recs.append(r)

FIELDS = ["mentee_id","rank","mentor_id","mentor_name","profession","mentor_rating",
          "organic_score","final_score","is_boosted","top_factor","profile_penalized",
          "w_skill_contrib","w_domain_contrib","w_exp_contrib","w_rating_contrib"]

with open(os.path.join(BASE_DIR,"recommendations.csv"),"w",newline="",encoding="utf-8") as f:
    wcsv = csv.DictWriter(f, fieldnames=FIELDS)
    wcsv.writeheader()
    wcsv.writerows(all_recs)

n_pen = sum(1 for r in all_recs if r["profile_penalized"])
n_bst = sum(1 for r in all_recs if r["is_boosted"])
top_f = Counter(r["top_factor"] for r in all_recs)

print(f"\nrecommendations.csv создан  ({len(all_recs)} строк)")
print(f"  С штрафом (нет навыков): {n_pen} ({n_pen/len(all_recs)*100:.1f}%)")
print(f"  С бустом:                {n_bst} ({n_bst/len(all_recs)*100:.1f}%)")
print("\nГлавный фактор:")
for factor, count in top_f.most_common():
    bar = chr(9608)*(count//80)
    print(f"  {factor:<10}  {count:>6}  ({count/len(all_recs)*100:.1f}%)  {bar}")

full  = [float(r["organic_score"]) for r in all_recs if not r["profile_penalized"]]
empty = [float(r["organic_score"]) for r in all_recs if r["profile_penalized"]]
if full and empty:
    print(f"\nЭффект штрафа:")
    print(f"  Полный профиль:    {sum(full)/len(full):.4f}")
    print(f"  Неполный профиль:  {sum(empty)/len(empty):.4f}")
    print(f"  Разница:           {sum(full)/len(full)-sum(empty)/len(empty):+.4f}")
print("\nТетрадка 3 завершена!")


Генерируем рекомендации для 5000 менти...
  0/5000...
  500/5000...
  1000/5000...
  1500/5000...
  2000/5000...
  2500/5000...
  3000/5000...
  3500/5000...
  4000/5000...
  4500/5000...

recommendations.csv создан  (25000 строк)
  С штрафом (нет навыков): 5334 (21.3%)
  С бустом:                7774 (31.1%)

Главный фактор:
  domain        8533  (34.1%)  ██████████████████████████████████████████████████████████████████████████████████████████████████████████
  skill         6570  (26.3%)  ██████████████████████████████████████████████████████████████████████████████████
  rating        5234  (20.9%)  █████████████████████████████████████████████████████████████████
  exp           4663  (18.7%)  ██████████████████████████████████████████████████████████

Эффект штрафа:
  Полный профиль:    0.8448
  Неполный профиль:  0.6463
  Разница:           +0.1985

Тетрадка 3 завершена!
